In [2]:
import time
import pandas as pd
import requests

# Daftar server Overpass API (otomatis pindah jika salah satu server sedang penuh)
overpass_urls = [
    "https://overpass.kumi.systems/api/interpreter",
    "https://lz4.overpass-api.de/api/interpreter",
    "https://overpass-api.de/api/interpreter",
]

# Menggunakan kode ISO Provinsi Bali (ID-BA) agar eksekusi kueri jauh lebih cepat
overpass_query = """
[out:json][timeout:90];
area["ISO3166-2"="ID-BA"]->.searchArea;
(
  node["amenity"="fuel"](area.searchArea);
  way["amenity"="fuel"](area.searchArea);
  relation["amenity"="fuel"](area.searchArea);
);
out center;
"""

# Header User-Agent wajib disertakan agar tidak diblokir oleh kebijakan Overpass API
headers = {
    "User-Agent": "BaliSPBUExtractor/1.0 (Data Analysis Project; Python)",
    "Accept": "application/json",
}

data = None
for url in overpass_urls:
    try:
        print(f"Mengambil data dari: {url} ...")
        response = requests.post(
            url, data={"data": overpass_query}, headers=headers, timeout=90
        )

        # Pastikan status HTTP 200 OK dan isi respons bukan halaman HTML error
        if response.status_code == 200 and response.text.strip().startswith(
            "{"
        ):
            data = response.json()
            print("Berhasil mendapatkan respons JSON!")
            break
        else:
            print(
                f"Server mengembalikan status {response.status_code}. Beralih ke server cadangan..."
            )
    except Exception as e:
        print(f"Gagal menghubungi {url}: {e}")
    time.sleep(2)

# Proses ekstraksi ke DataFrame jika data berhasil diambil
if data and "elements" in data:
    spbu_list = []
    for element in data["elements"]:
        lat = element.get("lat") or element.get("center", {}).get("lat")
        lon = element.get("lon") or element.get("center", {}).get("lon")
        tags = element.get("tags", {})

        spbu_list.append({
            "osm_id": element.get("id"),
            "name": tags.get("name", "SPBU (Tanpa Nama)"),
            "brand": tags.get("brand", tags.get("operator", "Unknown")),
            "latitude": lat,
            "longitude": lon,
            "address": tags.get("addr:street", tags.get("addr:full", "")),
            "opening_hours": tags.get("opening_hours", ""),
        })

    df = pd.DataFrame(spbu_list)
    df.to_csv("data_spbu_bali.csv", index=False)
    print(f"\nBerhasil menyimpan {len(df)} titik SPBU ke data_spbu_bali.csv")
else:
    print(
        "\nGagal mengambil data JSON. Pastikan koneksi internet tidak memblokir domain Overpass."
    )

Mengambil data dari: https://overpass.kumi.systems/api/interpreter ...
Server mengembalikan status 500. Beralih ke server cadangan...
Mengambil data dari: https://lz4.overpass-api.de/api/interpreter ...
Berhasil mendapatkan respons JSON!

Berhasil menyimpan 325 titik SPBU ke data_spbu_bali.csv


In [4]:
import folium
from folium.plugins import MarkerCluster

# Pastikan tidak ada koordinat kosong
df_clean = df.dropna(subset=["latitude", "longitude"]).copy()

# 1. Inisialisasi peta
center_lat = df_clean["latitude"].mean()
center_lon = df_clean["longitude"].mean()
m = folium.Map(location=[center_lat, center_lon], zoom_start=10, tiles="OpenStreetMap")

marker_cluster = MarkerCluster(name="Klaster SPBU").add_to(m)

base_domain = "https://www.google.com"

# 2. Iterasi setiap titik SPBU dan tambahkan tombol navigasi
for _, row in df_clean.iterrows():
    lat = row["latitude"]
    lon = row["longitude"]
    
    # Format URL pencarian titik koordinat & URL petunjuk arah (rute)
    url_lokasi = f"{base_domain}/maps/search/?api=1&query={lat},{lon}"
    url_rute = f"{base_domain}/maps/dir/?api=1&destination={lat},{lon}"
    
    popup_html = f"""
    <div style="font-family: Arial, sans-serif; font-size: 12px; width: 220px;">
        <b style="font-size: 13px;">{row['name']}</b><br>
        <b>Brand:</b> {row['brand']}<br>
        <b>Alamat:</b> {row['address'] if row['address'] else '-'}<br>
        <b>Jam Buka:</b> {row['opening_hours'] if row['opening_hours'] else '-'}<br>
        <hr style="margin: 8px 0;">
        <div style="display: flex; gap: 6px; justify-content: space-between;">
            <a href="{url_lokasi}" target="_blank" rel="noopener noreferrer" 
               style="background-color: #1a73e8; color: white; padding: 6px 8px; text-decoration: none; border-radius: 4px; font-weight: bold; text-align: center; flex: 1;">
               📍 Lokasi
            </a>
            <a href="{url_rute}" target="_blank" rel="noopener noreferrer" 
               style="background-color: #34a853; color: white; padding: 6px 8px; text-decoration: none; border-radius: 4px; font-weight: bold; text-align: center; flex: 1;">
               🚗 Rute
            </a>
        </div>
    </div>
    """
    
    brand_str = str(row["brand"]).lower()
    icon_color = "red" if "pertamina" in brand_str else "blue"
    
    folium.Marker(
        location=[lat, lon],
        popup=folium.Popup(popup_html, max_width=260),
        tooltip=f"{row['name']} (Klik untuk detail & rute)",
        icon=folium.Icon(color=icon_color, icon="gas-pump", prefix="fa")
    ).add_to(marker_cluster)

# 3. Simpan dan tampilkan peta
m.save("peta_spbu_bali.html")
m